# Interactive Exercise: Generate Forecasts and Prediction Intervals

In this exercise, you will use a selected ARIMA model to:

1. generate point forecasts;
2. produce 90%, 95%, and 99% prediction intervals;
3. visualize forecast uncertainty;
4. compare short- and long-horizon uncertainty; and
5. export the forecast results.

> **About the data:** The dataset is simulated for instructional purposes and does not contain official Illinois monthly expenditure figures.


## 1. Load the Required Packages


In [ ]:
packages <- c("readr", "dplyr", "ggplot2", "forecast", "tibble")

installed <- rownames(installed.packages())
missing_packages <- packages[!(packages %in% installed)]

if (length(missing_packages) > 0) {
  install.packages(missing_packages, repos = "https://cloud.r-project.org")
}

invisible(lapply(packages, library, character.only = TRUE))

## 2. Load the Simulated Monthly Expenditure Data


In [ ]:
df <- read_csv("../data/illinois_monthly_expenditures_simulated.csv")
df

## 3. Select an Expenditure Category

Change the value below to forecast another category.


In [ ]:
selected_series <- "education"

## 4. Prepare the Monthly Time Series


In [ ]:
plot_df <- df %>%
  filter(series == selected_series) %>%
  mutate(date = as.Date(date)) %>%
  arrange(date)

if (nrow(plot_df) == 0) {
  stop("The selected expenditure series was not found.")
}

start_year <- as.integer(format(min(plot_df$date), "%Y"))
start_month <- as.integer(format(min(plot_df$date), "%m"))

expenditure_ts <- ts(
  plot_df$expenditure,
  start = c(start_year, start_month),
  frequency = 12
)

expenditure_ts

## 5. Load or Refit the Selected Model

If the model-selection exercise was completed in the same Binder session, this notebook loads the saved model. Otherwise, it refits an automatic ARIMA model using BIC.


In [ ]:
model_path <- paste0(
  "../output/",
  selected_series,
  "_selected_arima_model.rds"
)

if (file.exists(model_path)) {
  selected_model <- readRDS(model_path)
  cat("Loaded the selected model from the output folder.\n")
} else {
  selected_model <- forecast::auto.arima(
    expenditure_ts,
    seasonal = FALSE,
    stepwise = FALSE,
    approximation = FALSE,
    test = "adf",
    ic = "bic"
  )
  cat(
    "No saved model was found.\n",
    "A new model was selected automatically using BIC.\n"
  )
}

selected_model

> **Reflection:** Try changing `ic = "bic"` to `ic = "aic"`. Does the selected model change? Why do you think different information criteria favor different models?

## 6. Choose the Forecast Horizon

Because the data are monthly:

- 12 periods represent one year;
- 24 periods represent two years;
- 36 periods represent three years.


In [ ]:
forecast_horizon <- 24

## 7. Generate Forecasts and Prediction Intervals

The `forecast()` function produces point forecasts together with 90%, 95%, and 99% prediction intervals.


In [ ]:
forecast_results <- forecast::forecast(
  selected_model,
  h = forecast_horizon,
  level = c(90, 95, 99)
)

forecast_results

## 8. Convert the Forecast Output to a Table


In [ ]:
forecast_table <- tibble(
  period = seq_len(forecast_horizon),
  point_forecast = as.numeric(forecast_results$mean),
  lower_90 = as.numeric(forecast_results$lower[, "90%"]),
  upper_90 = as.numeric(forecast_results$upper[, "90%"]),
  lower_95 = as.numeric(forecast_results$lower[, "95%"]),
  upper_95 = as.numeric(forecast_results$upper[, "95%"]),
  lower_99 = as.numeric(forecast_results$lower[, "99%"]),
  upper_99 = as.numeric(forecast_results$upper[, "99%"])
)

forecast_table

## 9. Add Forecast Dates


In [ ]:
last_date <- max(plot_df$date)

forecast_dates <- seq(
  from = seq(last_date, by = "month", length.out = 2)[2],
  by = "month",
  length.out = forecast_horizon
)

forecast_table <- forecast_table %>%
  mutate(date = forecast_dates) %>%
  select(
    date,
    point_forecast,
    lower_90,
    upper_90,
    lower_95,
    upper_95,
    lower_99,
    upper_99
  )

forecast_table

## 10. Plot the Forecast

The solid line shows the point forecast. The shaded areas show the 80% and 95% prediction intervals.


In [ ]:
forecast_plot <- forecast::autoplot(forecast_results) +
  labs(
    title = paste("ARIMA Forecast:", unique(plot_df$label)),
    subtitle = paste(
      forecast_horizon,
      "monthly periods with 90%, 95%, and 99% prediction intervals"
    ),
    x = "Year",
    y = "Expenditure ($ millions)"
  ) +
  theme_minimal(base_size = 13) +
  theme(
    plot.title = element_text(face = "bold"),
    panel.grid.minor = element_blank()
  )

forecast_plot

## 11. Compare Short- and Long-Horizon Uncertainty

Prediction intervals usually widen as the forecast horizon increases.


In [ ]:
interval_comparison <- forecast_table %>%
  mutate(
    width_90 = upper_90 - lower_90,
    width_95 = upper_95 - lower_95,
    width_99 = upper_99 - lower_99
  ) %>%
  slice(1, n()) %>%
  mutate(
    horizon = c(
      "First forecast period",
      "Final forecast period"
    )
  ) %>%
  select(
    horizon,
    date,
    point_forecast,
    width_90,
    width_95,
    width_99
  )

interval_comparison

## 12. Review the Forecast

Before using the forecast, consider whether:

- the projected path is consistent with recent expenditure behavior;
- the interval widths are reasonable;
- known policy changes are missing from the historical data;
- unusual historical events are affecting the model;
- the forecast horizon is appropriate for the planning decision.


## 13. Export the Forecast Results


In [ ]:
dir.create("../output", showWarnings = FALSE)

write_csv(
  forecast_table,
  paste0(
    "../output/",
    selected_series,
    "_arima_forecast.csv"
  )
)

ggsave(
  filename = paste0(
    "../output/",
    selected_series,
    "_arima_forecast.png"
  ),
  plot = forecast_plot,
  width = 8,
  height = 5,
  dpi = 300
)

cat("Forecast results and figure saved to the output folder.")

## Questions for Reflection

1. What path does the point forecast follow?

2. How do the 90%, 95%, and 99% prediction intervals differ?

3. Do the prediction intervals widen over the forecast horizon?

4. Does the forecast appear reasonable given the historical series?

5. Are there policy, demographic, or economic changes that the model cannot anticipate?

6. Would a shorter or longer forecast horizon be more useful for the budget decision?


## Try Another Expenditure Category

Return to the `selected_series` cell, choose another expenditure category, and rerun the notebook.


# Next Step

The forecast provides a model-based projection of future expenditures. Analysts should combine these results with policy knowledge and update the forecast as new data become available.
